# Project 2

### Group J

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC = "taxi-trips"

DB = "lakehouse.taxi"
BRONZE_TABLE = f"{DB}.bronze_trips"
SILVER_TABLE = f"{DB}.silver_trips"
GOLD_TABLE = f"{DB}.gold_peak_hour_trips"

BRONZE_CHECKPOINT = "/tmp/checkpoints/project2/bronze_trips"
SILVER_CHECKPOINT = "/tmp/checkpoints/project2/silver_trips"
GOLD_CHECKPOINT   = "/tmp/checkpoints/project2/gold_peak_hour_trips"

zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5, truncate=False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [3]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [4]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

raw_stream.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Bronze layer

In [5]:
# keep the Kafka event 'as-is' in Iceberg
# repo requirement: the bronze table must contain raw JSON rows

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    kafka_key STRING,
    raw_json STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP,
    bronze_ingested_at TIMESTAMP
)
USING iceberg
""")

DataFrame[]

In [6]:
# bronze layer dataframe
bronze_df = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_json"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
)

In [9]:
# bronze layer query
bronze_query = (
    bronze_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(processingTime="10 seconds")
    .toTable(BRONZE_TABLE)
)

In [10]:
# display bronze layer table
spark.sql(f"SELECT COUNT(*) AS bronze_rows FROM {BRONZE_TABLE}").show()

spark.sql(f"""
SELECT kafka_key, partition, offset, kafka_timestamp, raw_json
FROM {BRONZE_TABLE}
ORDER BY partition, offset
LIMIT 5
""").show(truncate=False)

+-----------+
|bronze_rows|
+-----------+
|    3475356|
+-----------+

+---------+---------+------+-----------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|kafka_key|partition|offset|kafka_timestamp        |raw_json                                                                                                                                                                                                                                                                                                                                            

## Silver layer

In [11]:
# parse JSON, cast types, clean bad rows, enrich with zone names, add the peak-hour flag

# create trip schema
trip_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("congestion_surcharge", DoubleType(), True)
])

# select pick-up zones
pickup_zones = (
    zones
    .select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("pickup_zone"),
        F.col("Borough").alias("pickup_borough")
    )
)

# select drop-off zones
dropoff_zones = (
    zones
    .select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Zone").alias("dropoff_zone"),
        F.col("Borough").alias("dropoff_borough")
    )
)

In [12]:
# parse the dataframe
parsed_df = (
    raw_stream
    .select(
        F.col("value").cast("string").alias("raw_json"),
        F.col("timestamp").alias("kafka_timestamp")
    )
    .select(
        F.from_json("raw_json", trip_schema).alias("j"),
        "raw_json",
        "kafka_timestamp"
    )
    .select("j.*", "raw_json", "kafka_timestamp")
)

# silver layer dataframe
silver_base_df = (
    parsed_df
    .withColumn("pickup_ts", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("dropoff_ts", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn(
        "is_peak_hour",
        (
            F.col("pickup_hour").between(7, 9) |
            F.col("pickup_hour").between(16, 19)
        )
    )
    .withColumn(
        "trip_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("VendorID").cast("string"), F.lit("")),
                F.coalesce(F.col("tpep_pickup_datetime"), F.lit("")),
                F.coalesce(F.col("tpep_dropoff_datetime"), F.lit("")),
                F.coalesce(F.col("PULocationID").cast("string"), F.lit("")),
                F.coalesce(F.col("DOLocationID").cast("string"), F.lit("")),
                F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
                F.coalesce(F.col("fare_amount").cast("string"), F.lit("")),
                F.coalesce(F.col("tip_amount").cast("string"), F.lit("")),
                F.coalesce(F.col("total_amount").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .filter(F.col("pickup_ts").isNotNull())
    .filter(F.col("dropoff_ts").isNotNull())
    .filter(F.col("PULocationID").isNotNull())
    .filter(F.col("DOLocationID").isNotNull())
    .filter(F.col("dropoff_ts") >= F.col("pickup_ts"))
    .filter(F.col("trip_distance") >= 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("tip_amount") >= 0)
    .filter(F.col("total_amount") >= 0)
    .withWatermark("pickup_ts", "45 days")
    .dropDuplicates(["trip_id"])
)

silver_df = (
    silver_base_df
    .join(pickup_zones, on="PULocationID", how="left")
    .join(dropoff_zones, on="DOLocationID", how="left")
    .withColumn("silver_ingested_at", F.current_timestamp())
    .select(
        "trip_id",
        "VendorID",
        "pickup_ts",
        "dropoff_ts",
        "pickup_hour",
        "is_peak_hour",
        "passenger_count",
        "trip_distance",
        "PULocationID",
        "pickup_zone",
        "pickup_borough",
        "DOLocationID",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type",
        "congestion_surcharge",
        "silver_ingested_at",
        "raw_json"
    )
)

In [13]:
# define silver layer table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    trip_id STRING,
    VendorID INT,
    pickup_ts TIMESTAMP,
    dropoff_ts TIMESTAMP,
    pickup_hour INT,
    is_peak_hour BOOLEAN,
    passenger_count INT,
    trip_distance DOUBLE,
    PULocationID INT,
    pickup_zone STRING,
    pickup_borough STRING,
    DOLocationID INT,
    dropoff_zone STRING,
    dropoff_borough STRING,
    fare_amount DOUBLE,
    tip_amount DOUBLE,
    total_amount DOUBLE,
    payment_type INT,
    congestion_surcharge DOUBLE,
    silver_ingested_at TIMESTAMP,
    raw_json STRING
)
USING iceberg
PARTITIONED BY (days(pickup_ts))
""")

DataFrame[]

In [24]:
# silver layer query
silver_query = (
    silver_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .trigger(processingTime="10 seconds")
    .toTable(SILVER_TABLE)
)

In [25]:
# display silver layer table
spark.sql(f"SELECT COUNT(*) AS silver_rows FROM {SILVER_TABLE}").show()

spark.sql(f"""
SELECT trip_id, pickup_ts, dropoff_ts, pickup_zone, dropoff_zone,
       pickup_hour, is_peak_hour, trip_distance, total_amount
FROM {SILVER_TABLE}
LIMIT 10
""").show(truncate=False)

spark.sql(f"""
SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT trip_id) AS distinct_trip_ids
FROM {SILVER_TABLE}
""").show()

+-----------+
|silver_rows|
+-----------+
|    1055546|
+-----------+

+----------------------------------------------------------------+-------------------+-------------------+-----------------------+----------------------------+-----------+------------+-------------+------------+
|trip_id                                                         |pickup_ts          |dropoff_ts         |pickup_zone            |dropoff_zone                |pickup_hour|is_peak_hour|trip_distance|total_amount|
+----------------------------------------------------------------+-------------------+-------------------+-----------------------+----------------------------+-----------+------------+-------------+------------+
|2c7cde27c41ddef362bdc0f0b82105698f24a3c271114c6af47026577294fdcf|2025-01-09 21:38:27|2025-01-09 21:46:30|Upper East Side South  |Upper East Side North       |21         |false       |1.5          |19.65       |
|6dc572cb29b432925d3d2d357dcb1e2adde54ece5dc0ad8812a76673a2f2a3aa|2025-01-09 21:0

## Gold layer

In [19]:
# create gold layer table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    pickup_zone STRING,
    pickup_hour INT,
    peak_trip_count BIGINT,
    non_peak_trip_count BIGINT
)
USING iceberg
""")

# streaming aggregation from silver; add 'is_peak_hour' column
gold_updates_df = (
    silver_df
    .groupBy("pickup_zone", "pickup_hour")
    .agg(
        F.sum(F.when(F.col("is_peak_hour") == True, 1).otherwise(0)).alias("peak_trip_count"),
        F.sum(F.when(F.col("is_peak_hour") == False, 1).otherwise(0)).alias("non_peak_trip_count")
    )
)

def upsert_gold(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    batch_spark = batch_df.sparkSession
    view_name = f"gold_updates_{batch_id}"

    (
        batch_df
        .select("pickup_zone", "pickup_hour", "peak_trip_count", "non_peak_trip_count")
        .createOrReplaceTempView(view_name)
    )

    batch_spark.sql(f"""
    MERGE INTO {GOLD_TABLE} AS t
    USING {view_name} AS s
    ON t.pickup_zone = s.pickup_zone
       AND t.pickup_hour = s.pickup_hour
    WHEN MATCHED THEN UPDATE SET
        t.peak_trip_count = s.peak_trip_count,
        t.non_peak_trip_count = s.non_peak_trip_count
    WHEN NOT MATCHED THEN INSERT (
        pickup_zone,
        pickup_hour,
        peak_trip_count,
        non_peak_trip_count
    )
    VALUES (
        s.pickup_zone,
        s.pickup_hour,
        s.peak_trip_count,
        s.non_peak_trip_count
    )
    """)

    batch_spark.catalog.dropTempView(view_name)

gold_query = (
    gold_updates_df.writeStream
    .outputMode("update")
    .option("checkpointLocation", GOLD_CHECKPOINT)
    .foreachBatch(upsert_gold)
    .start()
)

In [20]:
spark.sql(f"""
SELECT *
FROM {GOLD_TABLE}
ORDER BY pickup_hour, pickup_zone
LIMIT 20
""").show(truncate=False)

+-----------------------+-----------+---------------+-------------------+
|pickup_zone            |pickup_hour|peak_trip_count|non_peak_trip_count|
+-----------------------+-----------+---------------+-------------------+
|Alphabet City          |0          |0              |299                |
|Arrochar/Fort Wadsworth|0          |0              |1                  |
|Astoria                |0          |0              |19                 |
|Auburndale             |0          |0              |1                  |
|Baisley Park           |0          |0              |8                  |
|Battery Park           |0          |0              |2                  |
|Battery Park City      |0          |0              |59                 |
|Bay Ridge              |0          |0              |11                 |
|Bay Terrace/Fort Totten|0          |0              |1                  |
|Bedford                |0          |0              |10                 |
|Bellerose              |0          |0

In [21]:
spark.sql(f"""
SELECT
    ROUND(
        100.0 * SUM(peak_trip_count) / SUM(peak_trip_count + non_peak_trip_count),
        2
    ) AS peak_hour_share_pct
FROM {GOLD_TABLE}
""").show()

+-------------------+
|peak_hour_share_pct|
+-------------------+
|              38.63|
+-------------------+



In [29]:
# check if restarting the bronze job from the same checkpoint adds duplicate rows
before_count = spark.sql(f"SELECT COUNT(*) AS c FROM {BRONZE_TABLE}").collect()[0]["c"]
print("before_count =", before_count)

bronze_query.stop()

before_count = 3475356


In [31]:
# rerun the bronze layer query from same snapshot
bronze_query = (
    bronze_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", BRONZE_CHECKPOINT)
    .trigger(processingTime="10 seconds")
    .toTable(BRONZE_TABLE)
)

In [32]:
after_count = spark.sql(f"SELECT COUNT(*) AS c FROM {BRONZE_TABLE}").collect()[0]["c"]
print("after_count =", after_count)
print("duplicates_added =", after_count - before_count)

after_count = 3475356
duplicates_added = 0


In [27]:
# Iceberg snapshot history
spark.sql(f"""
SELECT *
FROM {GOLD_TABLE}.history
ORDER BY made_current_at DESC
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-03 14:55:29.87 |7387444524186307569|2645013020719896184|true               |
|2026-04-03 14:30:19.251|2645013020719896184|401475702803749137 |true               |
|2026-04-03 14:30:10.575|401475702803749137 |4874900899591244414|true               |
|2026-04-03 14:29:57.696|4874900899591244414|8757049672685916373|true               |
|2026-04-03 14:29:46.556|8757049672685916373|2973172003101512377|true               |
|2026-04-03 14:29:38.992|2973172003101512377|4993933397081828499|true               |
|2026-04-03 14:29:31.946|4993933397081828499|4191493839869939178|true               |
|2026-04-03 14:29:18.572|4191493839869939178|709715535015037940 |true               |
|2026-04-03 14:29:10.577|709715535015037940 |325694789